# AQSE — TQK8
8 qubit • 8 feature • VQC → K → Loss → QNG

Aprire dalla directory contenente `tqk8.py`. La prova numerica allegata è stata eseguita con il simulatore NumPy indipendente. Questo notebook non è stato eseguito con Qiskit nell’ambiente di generazione. Nessun job QPU viene inviato automaticamente.

In [ ]:
# Eseguire una volta nel proprio ambiente, se necessario:
# %pip install "qiskit>=2,<3" "numpy>=1.26,<3" "scikit-learn>=1.4,<2"

In [ ]:
import numpy as np
from tqk8 import (build_vqc, StateEngine, AngleScaler, demo_data,
                  alignment_loss, loss_grad_metric, fit_qng, overlap_circuit)
ENGINE = "qiskit"  # "numpy" per il simulatore indipendente
engine = StateEngine(ENGINE)

## 1. Circuito
Ordine fisico: RY(x) → RZ(α) → CZ → RY(β) → RZ(x).

Una trasformazione trainabile comune posta solo alla fine si cancellerebbe dal kernel di fedeltà.

In [ ]:
if ENGINE == "qiskit":
    circuit, data_parameters, trainable_parameters = build_vqc()
    print(circuit.draw("text", fold=140))
    print(circuit.count_ops())

## 2. Dati sintetici, split e normalizzazione
Per sensori reali sostituire lo split casuale con split temporale/per sessione e non lasciare finestre sovrapposte in training e test.

In [ ]:
from sklearn.model_selection import train_test_split
X, y = demo_data()
Xtr_raw, Xte_raw, ytr, yte = train_test_split(
    X, y, train_size=0.625, stratify=y, random_state=19)
scaler = AngleScaler.fit(Xtr_raw)
Xtr, Xte = scaler.transform(Xtr_raw), scaler.transform(Xte_raw)
theta0 = np.random.default_rng(7).uniform(-0.8, 0.8, 16)

## 3. Kernel, loss, gradiente e metrica
La matrice kernel ha dimensione B×B; la metrica g ha dimensione 16×16. Non sono la stessa matrice. La metrica è la media delle metriche Fubini–Study degli stati sul batch.

In [ ]:
loss0, gradient, g, K = loss_grad_metric(engine, Xtr, ytr, theta0)
print("Loss:", loss0)
print("Gradient:", gradient.shape)
print("Metric:", g.shape)
print("Kernel:", K.shape)
print("Autovalore minimo g:", np.linalg.eigvalsh(g)[0])

## 4. QNG e ritorno dei parametri al VQC
Si risolve (g+λI)v=∇L e si aggiorna θ←θ−ηv. La versione esatta usa line search e limita l’ampiezza del passo.

In [ ]:
theta, history = fit_qng(engine, Xtr, ytr, theta0, steps=15)
print("Loss finale:", alignment_loss(engine.gram(Xtr, theta), ytr)[0])

## 5. Chiusura classica
θ congelato; la SVC usa K(test,training). Non è necessario eseguire QNG per ogni predizione. Il baseline RBF usa gli stessi dati: la demo non dimostra un vantaggio quantistico.

In [ ]:
from sklearn.svm import SVC
from sklearn.metrics import balanced_accuracy_score
Ktr = engine.gram(Xtr, theta)
Kte = engine.gram(Xte, theta, Xtr)
model = SVC(kernel="precomputed", C=1.0).fit(Ktr, ytr)
pred = model.predict(Kte)
print("TQK:", balanced_accuracy_score(yte, pred))
rbf = SVC(kernel="rbf", C=1.0).fit(Xtr, ytr)
print("RBF:", balanced_accuracy_score(yte, rbf.predict(Xte)))

## 6. Conteggi a shot finiti, su simulatore ideale
Questo non include il rumore fisico della QPU.

In [ ]:
if ENGINE == "qiskit":
    from qiskit.primitives import StatevectorSampler
    from sampler_qng import SamplerOverlap
    executor = SamplerOverlap(StatevectorSampler(seed=42), shots=4096)
    k_shots = executor.probabilities([(Xtr[0], Xtr[1], theta, theta)])[0]
    print("Kernel esatto:", Ktr[0,1], "stima a shot:", k_shots)

## 7. Verifiche
Il test di confronto Qiskit/NumPy verrà eseguito se Qiskit è installato.

In [ ]:
import unittest, test_tqk8
suite = unittest.defaultTestLoader.loadTestsFromModule(test_tqk8)
unittest.TextTestRunner(verbosity=2).run(suite)